In [0]:
%pip install yfinance

In [0]:
# ── Imports ─────────────────────────────────────────────────────────────
import datetime
import time
import zoneinfo
import yfinance as yf
import pandas as pd

from pyspark.sql.functions import current_timestamp, to_date

# ── Config ─────────────────────────────────────────────────────────────
SYMBOL        = "^NSEI"
SESSION_START = datetime.time(9, 15)
SESSION_END   = datetime.time(15, 30)
IST           = zoneinfo.ZoneInfo("Asia/Kolkata")

INTERVAL_SEC  = 60
TABLE_NAME    = "nifty_live.session_ticks"

# ── Market Check ───────────────────────────────────────────────────────
def is_market_open():
    now = datetime.datetime.now(IST)
    if now.weekday() >= 5:
        return False
    return SESSION_START <= now.time() <= SESSION_END

# ── Fetch Latest Tick ──────────────────────────────────────────────────
def fetch_nifty():
    df = yf.download(SYMBOL, period="1d", interval="1m", progress=False)

    if df.empty:
        return None

    latest = df.iloc[[-1]].copy()

    # timezone safe
    latest.index = latest.index.tz_localize("UTC").tz_convert(IST)
    latest.index.name = "datetime_ist"

    latest = latest[["Open", "High", "Low", "Close"]]
    latest.columns = ["open", "high", "low", "close"]

    return latest

# ── Main Loop ──────────────────────────────────────────────────────────
print("🚀 Starting ingestion...")

while True:
    now = datetime.datetime.now(IST)

    # Stop if market closed
    if not is_market_open():
        print("❌ Market closed. Stopping ingestion.")
        break

    try:
        tick = fetch_nifty()

        if tick is None:
            print(f"[{now.strftime('%H:%M:%S')}] ⚠ No data")
            time.sleep(INTERVAL_SEC)
            continue

        # Remove duplicates
        tick = tick[~tick.index.duplicated(keep="last")]

        # Convert to Spark
        spark_df = spark.createDataFrame(tick.reset_index())

        # Add ingestion metadata
        spark_df = spark_df.withColumn("ingestion_time", current_timestamp())
        spark_df = spark_df.withColumn("date", to_date("datetime_ist"))

        # Write to Delta (single table)
        spark_df.write.format("delta") \
            .mode("append") \
            .partitionBy("date") \
            .saveAsTable(TABLE_NAME)

        print(f"[{now.strftime('%H:%M:%S')}] ✔ close={tick['close'].iloc[0]:.2f}")

    except Exception as e:
        print(f"[ERROR] {e}")

    time.sleep(INTERVAL_SEC)